# 04 · Avaliação
Fase 5 do CRISP-DM: os resultados respondem às perguntas do negócio? Quais são as limitações?

> ⚠️ **Dados fictícios.** Enquanto a coleta real não termina, este notebook usa
> `data/processed/desejos_demo.csv`, gerado por `src/exemplo.py`: 12 pessoas inventadas a partir de
> 4 estilos de desejo (gastronômico, tech, moda, pet), mais o perfil de exemplo "você".
> **Para usar os dados reais:** baixe o CSV anonimizado no app (*Meus Desejos → Baixar CSV anonimizado*),
> salve como `data/processed/desejos.csv` e troque `ARQUIVO` na célula abaixo. O resto roda igual.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # para importar src/

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette=["#6D4AFF", "#E8701F", "#17975F", "#C13B7A", "#2A7AB8", "#8C8A9E"])
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.precision", 2)

In [2]:
from sklearn.metrics import silhouette_score
from src.clustering import agrupar, escolher_k
perfis = pd.read_csv(Path.cwd().parent / "data" / "processed" / "perfis_demo.csv", index_col="usuario")
tabela = escolher_k(perfis)
k = int(tabela.loc[tabela["silhueta"].idxmax(), "k"])
rotulos, centros = agrupar(perfis, k)
s = silhouette_score(perfis, rotulos["cluster"])
print(f"k = {k} · silhueta = {s:.2f} · critério de sucesso (> 0,25): {'atingido' if s > 0.25 else 'NÃO atingido'}")
rotulos

k = 4 · silhueta = 0.57 · critério de sucesso (> 0,25): atingido


,usuario,cluster,perfil
0,u01,0,Gastronômico
1,u02,0,Gastronômico
2,u03,2,Tech & Games
3,u04,0,Gastronômico
4,u05,3,Moda & Beleza
5,u06,1,Pet lover
6,u07,2,Tech & Games
7,u08,2,Tech & Games
8,u09,0,Gastronômico
9,u10,3,Moda & Beleza


## 1. Os grupos fazem sentido?
Checagem qualitativa: o nome de cada grupo combina com o que as pessoas dele mais desejam?

In [3]:
from src.clustering import matriz_perfis
for _, r in rotulos.iterrows():
    top3 = perfis.loc[r["usuario"]].sort_values(ascending=False).head(3)
    print(f"{r['usuario']:<16} → {r['perfil']:<28} | top: " + ", ".join(f"{c} {v:.0%}" for c, v in top3.items()))

u01              → Gastronômico                 | top: comida 43%, bebidas 29%, calçados 14%
u02              → Gastronômico                 | top: comida 65%, bebidas 17%, veículos 13%
u03              → Tech & Games                 | top: eletrônicos 42%, brinquedos e games 35%, esporte e lazer 16%
u04              → Gastronômico                 | top: comida 46%, bebidas 25%, veículos 17%
u05              → Moda & Beleza                | top: calçados 37%, vestuário 30%, beleza 26%
u06              → Pet lover                    | top: animais 55%, esporte e lazer 18%, comida 14%
u07              → Tech & Games                 | top: eletrônicos 57%, brinquedos e games 22%, animais 13%
u08              → Tech & Games                 | top: brinquedos e games 43%, eletrônicos 21%, beleza 18%
u09              → Gastronômico                 | top: comida 42%, eletrônicos 38%, animais 15%
u10              → Moda & Beleza                | top: vestuário 36%, calçados 29%, beleza 25%
u11 

## 2. Estabilidade
Se o K-Means for rodado com outras sementes aleatórias, os grupos mudam? Grupos estáveis são mais confiáveis.

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
base = rotulos["cluster"].to_numpy()
ari = [adjusted_rand_score(base, KMeans(n_clusters=k, n_init=10, random_state=s).fit(perfis).labels_)
       for s in range(20)]
print(f"Concordância média com outras 20 sementes (ARI, 1 = idêntico): {np.mean(ari):.2f}")

Concordância média com outras 20 sementes (ARI, 1 = idêntico): 1.00


## 3. Acurácia do reconhecimento (a fazer com os dados reais)
Plano: separar **30 fotos reais**, rotular à mão o item correto e comparar com o 1º palpite do modelo
(e com os 3 primeiros, já que o app pede confirmação).

| Métrica | Fórmula | Meta |
|---|---|---|
| Acurácia top-1 | acertos no 1º palpite / 30 | ≥ 70% |
| Acurácia top-3 | item certo entre os 3 palpites / 30 | ≥ 90% |

```python
# quando houver o arquivo data/processed/avaliacao_clip.csv com colunas: foto, correto, palpite1, palpite2, palpite3
# av = pd.read_csv(...)
# top1 = (av.correto == av.palpite1).mean()
# top3 = av.apply(lambda r: r.correto in (r.palpite1, r.palpite2, r.palpite3), axis=1).mean()
```

## 4. Limitações
- **Dados fictícios:** os resultados acima demonstram o método, não descrevem pessoas reais.
- **Amostra pequena:** com poucas pessoas, o clustering tem pouca margem (k < número de pessoas) e a silhueta oscila.
- **Viés de quem usa o app:** colegas da mesma turma tendem a ter gostos e rotinas parecidos.
- **Reconhecimento:** o CLIP pode errar itens parecidos (pastel × pizza); a confirmação manual reduz isso.
- **Faixa de preço:** é uma referência por categoria, não o preço real do item fotografado.
- **OpenStreetMap:** a cobertura de lojas e abrigos varia por bairro.

## 5. Privacidade (LGPD): checklist
- [x] Consentimento explícito no cadastro
- [x] Foto salva sem EXIF/GPS; só a cidade é guardada
- [x] Comparação só com aceite das duas pessoas (RLS no banco)
- [x] CSV de análise anonimizado (`src/exportar.py`)
- [ ] Opção de apagar a conta e todos os desejos (próximo passo)

## 6. Conclusão e próximos passos
O método responde às perguntas do negócio: encontra perfis nomeáveis e mede o quanto duas pessoas combinam.
Próximos passos: coletar dados reais com a turma, rodar a avaliação do reconhecimento e refazer os notebooks 01–04
com `desejos.csv`.